In [7]:
### 1. SETUP — batch preprocessing across all subjects

import os, gc, time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")            # no GUI: QC figures are saved to disk, not shown
import matplotlib.pyplot as plt

import mne
from mne.preprocessing import ICA
from mne_bids import BIDSPath, read_raw_bids, get_entity_vals
from mne_icalabel import label_components
from autoreject import AutoReject, get_rejection_threshold

mne.set_log_level("ERROR")

BIDS_ROOT  = "/Users/elizabethkaplan/Desktop/ds007615/ds007615"          # dataset root
DERIV_ROOT = os.path.join(BIDS_ROOT, "derivatives", "preproc")  # outputs go here
TASK       = "rest"
ACQS       = ["ec", "eo"]        # eyes-closed + eyes-open resting-state

# Preproc parameters (identical to the single-subject version)
L_FREQ, H_FREQ  = 1.0, 40.0      # bandpass
NOTCH_FREQS     = [50, 100]      # line noise + harmonic (Norway = 50 Hz)
RESAMPLE_SFREQ  = 256
EOG_CHS         = ["HOG1", "HOG2", "VOG1", "VOG2"]
EPOCH_LEN       = 2.0
N_JOBS          = 10
MIN_KEPT_EPOCHS = 20             # fewer than this after cleaning -> flag "low_quality"
OVERWRITE       = False          # False = skip subjects already saved (safe to re-run)

os.makedirs(DERIV_ROOT, exist_ok=True)
print("Outputs ->", DERIV_ROOT)

Outputs -> /Users/elizabethkaplan/Desktop/ds007615/ds007615/derivatives/preproc


In [8]:
### 2. HELPERS — file checks + single-subject pipeline (wrapped for batch use)

def eeg_files(sub, acq):
    """Expected BrainVision triplet paths for one subject/acq."""
    base = os.path.join(BIDS_ROOT, f"sub-{sub}", "eeg",
                        f"sub-{sub}_task-{TASK}_acq-{acq}_eeg")
    return {ext: base + ext for ext in (".vhdr", ".eeg", ".vmrk")}

def has_required_files(sub, acq):
    """True only if all three BrainVision files exist. Also returns which are missing."""
    missing = [ext for ext, p in eeg_files(sub, acq).items() if not os.path.exists(p)]
    return (len(missing) == 0), missing

def preprocess_one(sub, acq):
    """Run the full pipeline for one subject/acq.
    Saves cleaned epochs (.fif) + QC pngs, and returns a result dict for the summary.
    Never raises: any failure is caught and recorded in the dict."""
    t0 = time.time()
    res = dict(subject=sub, acq=acq, status="ok", error="",
               n_epochs=np.nan, n_kept_thresh=np.nan, n_excluded_ics=np.nan,
               ic_label_counts="", n_final=np.nan, out_file="", seconds=np.nan)

    sub_dir = os.path.join(DERIV_ROOT, f"sub-{sub}")
    os.makedirs(sub_dir, exist_ok=True)
    out_fif = os.path.join(sub_dir, f"sub-{sub}_task-{TASK}_acq-{acq}_clean-epo.fif")
    res["out_file"] = out_fif

    if os.path.exists(out_fif) and not OVERWRITE:
        res["status"] = "skipped_exists"
        res["seconds"] = 0.0
        return res

    try:
        # LOAD
        bp = BIDSPath(subject=sub, task=TASK, acquisition=acq, suffix="eeg",
                      datatype="eeg", root=BIDS_ROOT)
        raw = read_raw_bids(bp, verbose=False)
        raw.load_data()

        # CHANNEL TYPES + MONTAGE
        raw.set_channel_types({ch: "eog" for ch in EOG_CHS if ch in raw.ch_names})
        if raw.get_montage() is None:
            raw.set_montage("standard_1005", on_missing="warn")

        # FILTER + RESAMPLE
        raw.notch_filter(freqs=NOTCH_FREQS, picks="eeg")
        raw.filter(l_freq=L_FREQ, h_freq=H_FREQ, picks=["eeg", "eog"], fir_design="firwin")
        raw.resample(RESAMPLE_SFREQ)

        # EPOCH + AMPLITUDE THRESHOLD
        events = mne.make_fixed_length_events(raw, start=0, stop=raw.times[-1],
                                              duration=EPOCH_LEN)
        epochs = mne.Epochs(raw, events=events, tmin=0.0, tmax=EPOCH_LEN,
                            baseline=None, picks="eeg", preload=True)
        res["n_epochs"] = len(epochs)
        reject = get_rejection_threshold(epochs, decim=1)
        epochs_clean = epochs.copy().drop_bad(reject=reject)
        res["n_kept_thresh"] = len(epochs_clean)
        if len(epochs_clean) == 0:
            raise RuntimeError("no epochs survived amplitude threshold")

        # ICA + ICLABEL
        epochs_clean.set_eeg_reference("average", projection=False)
        ica = ICA(n_components=0.99, method="infomax", max_iter="auto",
                  random_state=97, fit_params=dict(extended=True))
        ica.fit(epochs_clean)

        ic_out = label_components(epochs_clean, ica, method="iclabel")
        labels = ic_out["labels"]
        probs  = ic_out["y_pred_proba"]        # confidence of the winning label

        IC_THRESH = 0.80                       # only remove confident artifacts
        exclude_idx = [i for i, (lab, p) in enumerate(zip(labels, probs))
                       if lab not in ("brain", "other") and p >= IC_THRESH]

        res["ic_label_counts"] = ",".join(f"{k}:{v}" for k, v in Counter(labels).items())
        res["n_ics_total"]     = ica.n_components_     # so 12/55 vs 12/25 is visible
        res["n_excluded_ics"]  = len(exclude_idx)

        ica.exclude = exclude_idx
        epochs_ica = ica.apply(epochs_clean.copy())

        # AUTOREJECT
        ar = AutoReject(n_jobs=N_JOBS, random_state=42, picks="eeg", verbose=False)
        epochs_final, reject_log = ar.fit_transform(epochs_ica, return_log=True)
        res["n_final"] = len(epochs_final)
        if len(epochs_final) < MIN_KEPT_EPOCHS:
            res["status"] = "low_quality"

        # SAVE cleaned epochs
        epochs_final.save(out_fif, overwrite=True)

        # QC FIGURES (saved to disk; guarded so a plotting hiccup can't lose the subject)
        try:
            fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
            epochs_clean.compute_psd(fmax=45).plot(axes=axes[0], show=False)
            axes[0].set_title(f"sub-{sub} acq-{acq}  PSD pre-ICA")
            epochs_final.compute_psd(fmax=45).plot(axes=axes[1], show=False)
            axes[1].set_title("PSD post-ICA + AutoReject")
            fig.tight_layout()
            fig.savefig(os.path.join(sub_dir, f"sub-{sub}_task-{TASK}_acq-{acq}_qc-psd.png"), dpi=110)
            plt.close(fig)
            rl_fig = reject_log.plot("horizontal", show=False)
            rl_fig.savefig(os.path.join(sub_dir, f"sub-{sub}_task-{TASK}_acq-{acq}_qc-autoreject.png"), dpi=110)
            plt.close(rl_fig)
        except Exception as qc_e:
            res["error"] = f"(QC plot failed: {qc_e})"

    except Exception as e:
        res["status"] = "error"
        res["error"] = f"{type(e).__name__}: {e}"
    finally:
        res["seconds"] = round(time.time() - t0, 1)
        plt.close("all")
        gc.collect()
    return res

In [ ]:
##### 3. RUN BATCH — loop over every subject / acq

subjects = get_entity_vals(BIDS_ROOT, "subject")
print(f"Found {len(subjects)} subjects. TASK={TASK}, ACQS={ACQS}\n")

results = []
for n, sub in enumerate(subjects, 1):
    for acq in ACQS:
        ok, missing = has_required_files(sub, acq)
        if not ok:
            print(f"[{n}/{len(subjects)}] sub-{sub} acq-{acq}: SKIP — missing {missing}")
            results.append(dict(subject=sub, acq=acq, status="missing_files",
                                error=",".join(missing)))
            continue
        res = preprocess_one(sub, acq)
        print(f"[{n}/{len(subjects)}] sub-{sub} acq-{acq}: {res['status']:>14} | "
              f"final={res.get('n_final','?')} epochs | "
              f"excl={res.get('n_excluded_ics','?')} ICs | {res.get('seconds','?')}s"
              + (f" | {res['error']}" if res.get('error') else ""))
        results.append(res)

summary = pd.DataFrame(results)
summary_path = os.path.join(DERIV_ROOT, "processing_summary.csv")
summary.to_csv(summary_path, index=False)
print("\nSaved summary ->", summary_path)
print(summary["status"].value_counts())

Found 69 subjects. TASK=rest, ACQS=['ec', 'eo']

Estimating rejection dictionary for eeg


/var/folders/s6/21s_2dfj2f195fl1lglp7jrw0000gn/T/ipykernel_27932/1431960975.py:100: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


[1/69] sub-01 acq-ec:             ok | final=98 epochs | excl=0 ICs | 60.8s
Estimating rejection dictionary for eeg


/var/folders/s6/21s_2dfj2f195fl1lglp7jrw0000gn/T/ipykernel_27932/1431960975.py:100: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


[1/69] sub-01 acq-eo:             ok | final=65 epochs | excl=1 ICs | 36.2s
Estimating rejection dictionary for eeg


...

[43/69] sub-43 acq-eo:             ok | final=98 epochs | excl=2 ICs | 128.4s
Estimating rejection dictionary for eeg


/var/folders/s6/21s_2dfj2f195fl1lglp7jrw0000gn/T/ipykernel_27932/1431960975.py:100: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


[44/69] sub-44 acq-ec:             ok | final=122 epochs | excl=3 ICs | 190.2s
Estimating rejection dictionary for eeg


/var/folders/s6/21s_2dfj2f195fl1lglp7jrw0000gn/T/ipykernel_27932/1431960975.py:100: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


[44/69] sub-44 acq-eo:             ok | final=89 epochs | excl=5 ICs | 107.0s
Estimating rejection dictionary for eeg


In [7]:
### 4. REVIEW — subjects that need a human eyeball

summary = pd.read_csv(os.path.join(DERIV_ROOT, "processing_summary.csv"))

needs_attention = summary[summary["status"].isin(["error", "low_quality", "missing_files"])]
print(f"{len(needs_attention)} of {len(summary)} runs need attention:\n")
needs_attention

1 of 14 runs need attention:



,subject,acq,status,error,n_epochs,n_kept_thresh,n_excluded_ics,ic_label_counts,n_final,out_file,seconds
13,7,eo,missing_files,.vmrk,NaN,NaN,NaN,NaN,NaN,NaN,NaN
